# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [34]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Respuesta:**

- **Gymnasium:** proporciona el entorno (la dinámica del mundo): estados, espacio de acciones, la función de transición implícita, la función de recompensa y el render. Es la fuente de experiencias reales (muestras `(s,a,r,s')`).
- **El agente:** no está en una clase separada; queda representado por la `Q-table` junto con las funciones de política (`epsilon_greedy_policy`, `greedy_policy`) y el bucle de entrenamiento `train_q_learning`. Es decir, el agente = `Qtable` + funciones que seleccionan y actualizan acciones.

**Notas:**

- Gymnasium = entorno (modelo/ejecutor).
- Agente = datos (Q) + algoritmos (política, actualización).

## 2. Crear FrozenLake


In [35]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
   - Igual al número de estados del ambiente: `env.observation_space.n` (para `4x4` → 16 filas).
2. ¿Cuántas columnas?
   - Igual al número de acciones: `env.action_space.n` (FrozenLake → 4 columnas).
3. ¿Qué representa una celda $Q[s,a]$?

**Respuesta / discusión:**

- `Q[s,a]` es la estimación del valor (action-value) de tomar la acción `a` en el estado `s`: el retorno acumulado esperado (descontado) si desde `s` ejecutamos `a` y seguimos la política óptima después.
- Es una estimación que el agente actualiza con muestras y errores TD; no es la recompensa inmediata, sino el valor esperado a largo plazo.

- Ejemplo: `Q.shape == (env.observation_space.n, env.action_space.n)`

In [36]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- Que `Q` empiece en cero no implica que las acciones sean "malas": significa que el agente **todavía no sabe**; no hay información sobre retornos futuros.
- Si varias acciones tienen el mismo valor máximo, se produce un empate; la política greedy implementada desempata aleatoriamente entre las mejores (por eso `greedy_policy` usa `np.random.choice`). Esto puede inducir comportamiento exploratorio implícito en los empates; en la práctica conviene explorar explícitamente con $
\epsilon$-greedy para descubrir diferencias reales.

- Con `Q=0` inicialmente, la decisión inicial es arbitraria hasta recibir experiencias.

## 3. Ejecutar una transición


In [37]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 1
reward     = 0
next_state = 4
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$(s, a, r, s')$$

(usa los valores impresos por la celda para completar los números concretos cuando ejecutes)

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?

**Respuesta:**

- En Value Iteration trabajamos con el **modelo** (probabilidades de transición $P(s'|s,a)$ y recompensas $R(s,a)$), no con **muestras individuales**. Por lo tanto, cuando hablamos de experiencia real nos referimos a la muestra concreta $(s,a,r,s')$, que **no es necesaria** para Value Iteration clásica: VI no usa experiencias individuales, usa el modelo completo.



## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [38]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [39]:
# Momento 1: sin entrenamiento (random walk) — usar versión sin render para evitar dependencia de pygame
states_random, reward_random = run_episode_no_render(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
print("Estados recorridos (primeros):", states_random[:10])

Recompensa total: 0.0
Estados recorridos (primeros): [0, 0, 0, 0, 0, 0, 4, 4, 4, 5]


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [40]:
# Momento 2 — Después de pocas iteraciones (usar versión sin render)
states_early, reward_early = run_episode_no_render(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
print("Estados recorridos (primeros):", states_early[:10])

Recompensa total: 0.0
Estados recorridos (primeros): [0, 0, 0, 0, 0, 1, 5]


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [41]:
# Momento 3 — Agente entrenado (usar versión sin render)
states_trained, reward_trained = run_episode_no_render(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
print("Estados recorridos (primeros):", states_trained[:10])

Recompensa total: 1.0
Estados recorridos (primeros): [0, 1, 2, 6, 10, 14, 15]


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

- Internamente la `Q-table` dejó de estar vacía y acumuló información: los valores `Q(s,a)` se actualizaron mediante las reglas TD hacia los `td_target = r + \gamma \max_{a'} Q(s',a')`.
- Esos cambios hacen que la política greedy favorezca acciones con mayor retorno esperado; es decir, el agente pasa de elegir al azar a preferir acciones que históricamente conducen a mejores recompensas.
- Además, la reducción de $\epsilon$ en el entrenamiento hace que la exploración disminuya y la explotación de la mejor política aprendida aumente con el tiempo.

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** 0  

**Valores Q (ejemplo de interpretación):**

- Left: `Q[0,0]`  
- Down: `Q[0,1]`  
- Right: `Q[0,2]`  
- Up: `Q[0,3]`  

(Ejecuta la celda que imprime `Q` para ver los valores numéricos reales tras el entrenamiento.)

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

- Se seleccionará la acción con el mayor valor entre las cuatro anteriores (si hay empate, la implementación desempata aleatoriamente).

**Interpretación:**

- El valor mayor indica la acción que el agente estima como con mayor retorno esperado desde ese estado. Elegir `\arg\max` equivale a seguir la política greedy derivada de `Q`.

## 5. Evaluar la política aprendida


In [42]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

- Durante la evaluación queremos medir la calidad de la política aprendida, es decir, su rendimiento determinístico aplicando la mejor acción conocida. Si exploráramos (elegir acciones al azar) introduciríamos variabilidad y no evaluaríamos la política sino una mezcla de política+ruido.
- Explorar es útil durante el entrenamiento para descubrir información; durante la evaluación buscamos una estimación estable del comportamiento del agente sin exploración.

## 6. Experimento: FrozenLake estocástico


In [43]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.432
Std reward : 0.495


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

- Con `is_slippery=False` las transiciones son determinísticas: la misma acción en un estado lleva siempre al mismo `s'` (siempre que la función de transición lo defina así).
- Con `is_slippery=True` las transiciones son estocásticas: la acción tomada puede llevar a distintos `s'` según probabilidades (el agente "resbala" y puede acabar en un estado distinto del esperado).

¿Qué cambia en la **ecuación de Q-Learning**?

- La forma algebraica de la actualización no cambia: seguimos usando
  $$\text{td\_target} = r + \gamma \max_{a'} Q(s',a')$$
  y
  $$Q(s,a) \leftarrow Q(s,a) + \alpha (\text{td\_target} - Q(s,a)).$$
- Lo que cambia es que las muestras $(s,a,r,s')$ son más ruidosas y, por tanto, el aprendizaje suele requerir más episodios y puede converger más lentamente. En práctica hay que ajustar hiperparámetros (tasa de aprendizaje, número de episodios, decaimiento de $\epsilon$) para manejar la mayor varianza.

**Discusión:**
- Ambiente: transiciones determinísticas → estocásticas; mayor incertidumbre.
- Algoritmo: misma ecuación, pero mayor varianza en las actualizaciones y necesidad de más experiencia para estimar valores esperados.

# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

- `Q(s,a)` almacena la estimación del **valor acción**: el retorno acumulado esperado (posiblemente descontado) al tomar la acción `a` en el estado `s` y seguir la política óptima después. Es una estimación que se actualiza con errores TD.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

- Proviene de la `Q-table`: es el mayor valor estimado entre todas las acciones posibles en el estado sucesor `s'`. Representa la mejor recompensa futura estimada desde `s'` según la información actual del agente.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

- Para balancear **exploración** (probar acciones menos conocidas para obtener información) y **explotación** (usar las mejores acciones conocidas). Sin exploración el agente puede quedarse en soluciones subóptimas por falta de información.

**4. ¿Por qué Q-Learning es model-free?**

- Porque actualiza `Q(s,a)` únicamente a partir de muestras de experiencia $(s,a,r,s')` y no requiere conocer ni estimar explícitamente el modelo del ambiente (las probabilidades de transición $P(s'|s,a)$ ni la función de recompensa analítica). Aprendemos valores directamente de la interacción.

In [44]:
# Ejecutar episodios sin render para evitar dependencia de pygame

def run_episode_no_render(env, Q=None, random_policy=False, max_steps=100, seed=None):
    state, _ = env.reset(seed=seed)
    total_reward = 0.0
    states = [state]

    for _ in range(max_steps):
        if random_policy or Q is None:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        states.append(next_state)
        state = next_state

        if terminated or truncated:
            break

    return states, total_reward

# Episodio 1: random
states_r, rew_r = run_episode_no_render(env, Q_initial, random_policy=True, seed=7)
print(f"Random policy reward: {rew_r}")

# Episodio 2: early (greedy)
states_e, rew_e = run_episode_no_render(env, Q_early, random_policy=False, seed=7)
print(f"Early policy reward: {rew_e}")

# Episodio 3: trained (greedy)
states_t, rew_t = run_episode_no_render(env, Q_trained, random_policy=False, seed=7)
print(f"Trained policy reward: {rew_t}")

# Mostrar primeras transiciones como ejemplo
print('\nPrimeras 10 transiciones (estados) del episodio entrenado:', states_t[:10])

Random policy reward: 0.0
Early policy reward: 0.0
Trained policy reward: 1.0

Primeras 10 transiciones (estados) del episodio entrenado: [0, 4, 8, 9, 10, 14, 15]
